In [0]:
%run ../config_setup

# Silver 03 — Reference Data (Owner, Operator, Model)

SCD1-style reference tables: `owner`, `operator`, `machine_model` ke latest states env-configured silver tables me overwrite mode se likhta hai — history maintain nahi hoti.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, get_json_object, expr

spark = SparkSession.builder.getOrCreate()

# Owner ke liye custom function (owner_type ke saath)
def process_owner():
    df = spark.readStream.table(f"{BRONZE}.rds_cdc").filter(col("table") == "owner").select(
        col("_lsn"), col("_change_type"),
        get_json_object("after", "$.owner_id").alias("owner_id"),
        get_json_object("after", "$.owner_name").alias("owner_name"),
        get_json_object("after", "$.owner_type").alias("owner_type")
    )
    
    def upsert(microBatchDF, batchId):
        if microBatchDF.count() == 0: return
        latest = microBatchDF.withColumn("rn", expr("row_number() over (partition by owner_id order by _lsn desc)")).filter("rn = 1").drop("rn")
        latest.filter(col("_change_type") != "delete").drop("_lsn", "_change_type") \
              .write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.owner")
              
    df.writeStream.foreachBatch(upsert).option("checkpointLocation", f"{CHECKPOINTS}/silver_owner").trigger(availableNow=True).start().awaitTermination()

# Operator (basic)
def process_operator():
    df = spark.readStream.table(f"{BRONZE}.rds_cdc").filter(col("table") == "operator").select(
        col("_lsn"), col("_change_type"),
        get_json_object("after", "$.operator_id").alias("operator_id"),
        get_json_object("after", "$.operator_name").alias("operator_name")
    )
    def upsert(microBatchDF, batchId):
        if microBatchDF.count() == 0: return
        latest = microBatchDF.withColumn("rn", expr("row_number() over (partition by operator_id order by _lsn desc)")).filter("rn = 1").drop("rn")
        latest.filter(col("_change_type") != "delete").drop("_lsn", "_change_type").write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.operator")
    df.writeStream.foreachBatch(upsert).option("checkpointLocation", f"{CHECKPOINTS}/silver_operator").trigger(availableNow=True).start().awaitTermination()

# Machine Model (basic)
def process_machine_model():
    df = spark.readStream.table(f"{BRONZE}.rds_cdc").filter(col("table") == "machine_model").select(
        col("_lsn"), col("_change_type"),
        get_json_object("after", "$.model_id").alias("model_id"),
        get_json_object("after", "$.model_name").alias("model_name")
    )
    def upsert(microBatchDF, batchId):
        if microBatchDF.count() == 0: return
        latest = microBatchDF.withColumn("rn", expr("row_number() over (partition by model_id order by _lsn desc)")).filter("rn = 1").drop("rn")
        latest.filter(col("_change_type") != "delete").drop("_lsn", "_change_type").write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.machine_model")
    df.writeStream.foreachBatch(upsert).option("checkpointLocation", f"{CHECKPOINTS}/silver_machine_model").trigger(availableNow=True).start().awaitTermination()

# Sab run karo
process_owner()
process_operator()
process_machine_model()

print("✅ Reference Data (Owner, Operator, Model) updated successfully!")